<a href="https://colab.research.google.com/github/alissonfilipe/SparkReadDrive/blob/main/tratamentoSpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalar Java e Spark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz
!pip install -q findspark

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init()

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("YouTubeDataCleaning") \
    .getOrCreate()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content/drive/MyDrive/USvideos.csv

/content/drive/MyDrive/USvideos.csv


In [21]:
# Caminho do arquivo no Google Drive
caminho = "/content/drive/MyDrive/videos-stats.csv"

# Leitura do CSV no PySpark
df_video = (
    spark.read
    .option("header", True)        # usa a primeira linha como cabeçalho
    .option("inferSchema", True)   # infere automaticamente os tipos das colunas
    .csv(caminho)
)

# Visualizar as primeiras linhas
df_video.show(5)

# Verificar o esquema inferido
df_video.printSchema()

+---+--------------------+-----------+-------------------+-------+-------+--------+---------+
|_c0|               Title|   Video ID|       Published At|Keyword|  Likes|Comments|    Views|
+---+--------------------+-----------+-------------------+-------+-------+--------+---------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|2022-08-23 00:00:00|   tech| 3407.0|   672.0| 135612.0|
|  1|The most EXPENSIV...|b3x28s61q3c|2022-08-24 00:00:00|   tech|76779.0|  4306.0|1758063.0|
|  2|My New House Gami...|4mgePWWCAmA|2022-08-23 00:00:00|   tech|63825.0|  3338.0|1564007.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|2022-08-23 00:00:00|   tech|71566.0|  1426.0| 922918.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|2022-08-08 00:00:00|   tech|96513.0|  5155.0|1855644.0|
+---+--------------------+-----------+-------------------+-------+-------+--------+---------+
only showing top 5 rows

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Publis

In [22]:
df_video = df_video.fillna(
    {
        "Likes": 0,
        "Comments": 0,
        "Views": 0
    }
)

# Conferir se funcionou
df_video.select("Likes", "Comments", "Views").show(5)

+-------+--------+---------+
|  Likes|Comments|    Views|
+-------+--------+---------+
| 3407.0|   672.0| 135612.0|
|76779.0|  4306.0|1758063.0|
|63825.0|  3338.0|1564007.0|
|71566.0|  1426.0| 922918.0|
|96513.0|  5155.0|1855644.0|
+-------+--------+---------+
only showing top 5 rows



In [23]:
# Caminho do arquivo no Google Drive
caminho_comentarios = "/content/drive/MyDrive/comments.csv"

# Leitura do CSV no PySpark
df_comentario = (
    spark.read
    .option("header", True)        # usa a primeira linha como nome das colunas
    .option("inferSchema", True)   # infere automaticamente os tipos
    .csv(caminho_comentarios)
)

# Visualizar os dados
df_comentario.show(5)

# Verificar os tipos das colunas
df_comentario.printSchema()

+---+-----------+--------------------+-----+---------+
|_c0|   Video ID|             Comment|Likes|Sentiment|
+---+-----------+--------------------+-----+---------+
|  0|wAZZ-UWGVHI|Let's not forget ...| 95.0|      1.0|
|  1|wAZZ-UWGVHI|Here in NZ 50% of...| 19.0|      0.0|
|  2|wAZZ-UWGVHI|I will forever ac...|161.0|      2.0|
|  3|wAZZ-UWGVHI|Whenever I go to ...|  8.0|      0.0|
|  4|wAZZ-UWGVHI|Apple Pay is so c...| 34.0|      2.0|
+---+-----------+--------------------+-----+---------+
only showing top 5 rows

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [24]:
# Quantidade de registros em cada dataframe
qtd_videos = df_video.count()
qtd_comentarios = df_comentario.count()

print("Quantidade de registros em df_video:", qtd_videos)
print("Quantidade de registros em df_comentario:", qtd_comentarios)

Quantidade de registros em df_video: 1881
Quantidade de registros em df_comentario: 30036


In [25]:
from pyspark.sql.functions import col

df_video = df_video.filter(col("Video ID").isNotNull())
df_comentario = df_comentario.filter(col("Video ID").isNotNull())

In [26]:
qtd_videos = df_video.count()
qtd_comentarios = df_comentario.count()

print("Quantidade de registros em df_video após limpeza:", qtd_videos)
print("Quantidade de registros em df_comentario após limpeza:", qtd_comentarios)

Quantidade de registros em df_video após limpeza: 1881
Quantidade de registros em df_comentario após limpeza: 22555


In [28]:
df_video = df_video.dropDuplicates(["Video ID"])

In [29]:
print("Quantidade de registros em df_video após remover duplicados:", df_video.count())

Quantidade de registros em df_video após remover duplicados: 1869


In [30]:
from pyspark.sql.functions import col

df_video = (
    df_video
    .withColumn("Likes", col("Likes").cast("int"))
    .withColumn("Comments", col("Comments").cast("int"))
    .withColumn("Views", col("Views").cast("int"))
)

In [31]:
df_video.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: timestamp (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)



In [32]:
from pyspark.sql.functions import col

df_comentario = (
    df_comentario
    .withColumn("Likes", col("Likes").cast("int"))
    .withColumn("Sentiment", col("Sentiment").cast("int"))
    .withColumnRenamed("Likes", "Likes Comment")
)

In [33]:
df_comentario.printSchema()
df_comentario.show(5)

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: integer (nullable = true)
 |-- Sentiment: integer (nullable = true)

+---+-----------+--------------------+-------------+---------+
|_c0|   Video ID|             Comment|Likes Comment|Sentiment|
+---+-----------+--------------------+-------------+---------+
|  0|wAZZ-UWGVHI|Let's not forget ...|           95|        1|
|  1|wAZZ-UWGVHI|Here in NZ 50% of...|           19|        0|
|  2|wAZZ-UWGVHI|I will forever ac...|          161|        2|
|  3|wAZZ-UWGVHI|Whenever I go to ...|            8|        0|
|  4|wAZZ-UWGVHI|Apple Pay is so c...|           34|        2|
+---+-----------+--------------------+-------------+---------+
only showing top 5 rows



In [34]:
from pyspark.sql.functions import col

df_video = df_video.withColumn(
    "Interaction",
    col("Likes") + col("Comments") + col("Views")
)

In [35]:
df_video.select("Video ID", "Likes", "Comments", "Views", "Interaction").show(5)

+-----------+------+--------+--------+-----------+
|   Video ID| Likes|Comments|   Views|Interaction|
+-----------+------+--------+--------+-----------+
|115amzVdV44|910553|   81975|52061447|   53053975|
|m7Jw3a7CpNA| 32092|     181|  425850|     458123|
|V6hofBnlJLY|    16|       3|   83121|      83140|
|Rk4bAofG8xE|  8534|     517|  254621|     263672|
|zdsdEVngg7Y| 90491|    1633| 2854238|    2946362|
+-----------+------+--------+--------+-----------+
only showing top 5 rows



In [36]:
from pyspark.sql.functions import col, to_date

df_video = df_video.withColumn(
    "Published At",
    to_date(col("Published At"))
)

In [37]:
df_video.printSchema()
df_video.select("Video ID", "Published At").show(5)

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)

+-----------+------------+
|   Video ID|Published At|
+-----------+------------+
|115amzVdV44|  2020-08-18|
|m7Jw3a7CpNA|  2021-12-03|
|V6hofBnlJLY|  2022-08-18|
|Rk4bAofG8xE|  2022-08-18|
|zdsdEVngg7Y|  2021-04-22|
+-----------+------------+
only showing top 5 rows



In [38]:
from pyspark.sql.functions import col, year

df_video = df_video.withColumn(
    "Year",
    year(col("Published At"))
)

In [39]:
df_video.select("Published At", "Year").show(5)

+------------+----+
|Published At|Year|
+------------+----+
|  2020-08-18|2020|
|  2021-12-03|2021|
|  2022-08-18|2022|
|  2022-08-18|2022|
|  2021-04-22|2021|
+------------+----+
only showing top 5 rows



In [40]:
df_join_video_comments = df_video.join(
    df_comentario,
    on="Video ID",
    how="left"
)

In [41]:
df_join_video_comments.show(5)
df_join_video_comments.printSchema()

+-----------+---+--------------------+------------+-------+------+--------+--------+-----------+----+----+--------------------+-------------+---------+
|   Video ID|_c0|               Title|Published At|Keyword| Likes|Comments|   Views|Interaction|Year| _c0|             Comment|Likes Comment|Sentiment|
+-----------+---+--------------------+------------+-------+------+--------+--------+-----------+----+----+--------------------+-------------+---------+
|--ZI0dSbbNU|986|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|9584|I Love that he us...|            1|        2|
|--ZI0dSbbNU|986|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|9583|something that ne...|            1|        2|
|--ZI0dSbbNU|986|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|9582|his food always l...|            3|        2|
|--ZI0dSbbNU|986|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   1

In [42]:
# Caminho do arquivo no Google Drive
caminho_usvideos = "/content/drive/MyDrive/USvideos.csv"

# Leitura do CSV
df_us_videos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(caminho_usvideos)
)

# Visualizar os dados
df_us_videos.show(5)

# Verificar o esquema
df_us_videos.printSchema()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [43]:
df_join_video_usvideos = df_video.join(
    df_us_videos,
    on="Title",
    how="left"
)

# Visualizar os dados
df_join_video_usvideos.show(5)

# Verificar estrutura
df_join_video_usvideos.printSchema()

+--------------------+----+-----------+------------+-------+------+--------+--------+-----------+----+--------+-------------+-------------+-----------+------------+----+-----+-----+--------+-------------+--------------+-----------------+----------------+----------------------+-----------+
|               Title| _c0|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|video_id|trending_date|channel_title|category_id|publish_time|tags|views|likes|dislikes|comment_count|thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|description|
+--------------------+----+-----------+------------+-------+------+--------+--------+-----------+----+--------+-------------+-------------+-----------+------------+----+-----+-----+--------+-------------+--------------+-----------------+----------------+----------------------+-----------+
|Celebrating My 40...| 993|-64r1hcxtV4|  2022-05-30|mukbang| 45628|   17264| 5283664|    5346556|2022|    null|         null|     

In [44]:
from pyspark.sql.functions import col, sum, when

df_video.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_video.columns
]).show()

+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|_c0|Title|Video ID|Published At|Keyword|Likes|Comments|Views|Interaction|Year|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|  0|    0|       0|           0|      0|    0|       0|    0|          0|   0|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+



In [45]:
df_video = df_video.drop("_c0")

In [46]:
caminho_saida = "/content/drive/MyDrive/videos-tratados-parquet"

df_video.write \
    .mode("overwrite") \
    .parquet(caminho_saida)

In [47]:
df_verificacao = spark.read.parquet(caminho_saida)
df_verificacao.show(5)
df_verificacao.printSchema()

+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|2022|
|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|mukbang| 45628|   17264| 5283664|    5346556|2022|
|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|physics| 10959|     525|  844015|     855499|2017|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
only showing top 5 rows

root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Publis

In [48]:
df_join_video_comments = df_join_video_comments.drop("_c0")

In [49]:
caminho_saida = "/content/drive/MyDrive/videos-comments-tratados-parquet"

df_join_video_comments.write \
    .mode("overwrite") \
    .parquet(caminho_saida)

In [50]:
df_verificacao = spark.read.parquet(caminho_saida)
df_verificacao.show(5)
df_verificacao.printSchema()

+-----------+--------------------+------------+-------+------+--------+--------+-----------+----+--------------------+-------------+---------+
|   Video ID|               Title|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|             Comment|Likes Comment|Sentiment|
+-----------+--------------------+------------+-------+------+--------+--------+-----------+----+--------------------+-------------+---------+
|--ZI0dSbbNU|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|I Love that he us...|            1|        2|
|--ZI0dSbbNU|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|something that ne...|            1|        2|
|--ZI0dSbbNU|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|his food always l...|            3|        2|
|--ZI0dSbbNU|ASMR MUKBANG DOUB...|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|The burgers looks...|            7|        2|